<a href="https://colab.research.google.com/github/FruitNinja156/ai-coursework-code/blob/main/Major_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Config Section!

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
import random
import re
import unicodedata
import numpy as np
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from collections import Counter
from pathlib import Path
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
import re
import unicodedata
import numpy as np
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from collections import Counter
from pathlib import Path
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from gensim.models.coherencemodel import CoherenceModel

@dataclass(frozen=True)
class Paths:
    project_root: Path = Path.cwd()
    data_dir: Path = Path("/content/drive/MyDrive/Major Project/")
    output_dir: Path = data_dir / "outputs"

    train_file: Path = data_dir / "train.txt"
    label_file: Path = data_dir / "label.txt"
    test_file: Path = data_dir / "test.txt"

    topic_output: Path = output_dir / "topic.txt"
    emotion_output: Path = output_dir / "emotion.txt"

@dataclass(frozen=True)
class TopicConfig:
    num_topics: int = 10
    top_n_words: int = 5

    min_token_len: int = 3
    no_below:int = 2
    no_above:float = 0.6
    passes: int = 25
    iterations: int = 400
    chunksize: int = 200

    alpha:str = "auto"
    eta:str= "auto"
    eval_every:None =None
    coherence_metric: str = "c_v"

    random_state: int = 42

@dataclass(frozen=True)
class EmotionConfig:
    emotions: Tuple[str, ...] = (
        "anger", "disgust", "fear", "joy", "sadness", "surprise"
    )
    num_classes: int = 6
    min_word_freq: int = 1
    max_vocab_size: int = 20000
    pad_token: str = "<pad>"
    unk_token: str = "<unk>"
    max_seq_len: int = 32
    embedding_dim: int = 128
    lstm_hidden: int = 128
    lstm_layers: int = 1
    attention_dim: int = 96
    dropout: float = 0.4
    batch_size: int = 32
    epochs: int = 60
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 5.0
    val_split: float = 0.15
    patience: int = 8
    label_smoothing_eps: float = 1e-4
    ensemble_seeds: Tuple[int, ...] = (13, 42, 2024)
    loss_fn: str = "kl_div"

@dataclass(frozen=True)
class Config:
    paths: Paths = field(default_factory=Paths)
    topic: TopicConfig = field(default_factory=TopicConfig)
    emotion: EmotionConfig = field(default_factory=EmotionConfig)


CFG = Config()

STOPWORDS: frozenset = frozenset("""
a about above after again against all am an and any are aren as at
be because been before being below between both but by
can cannot could couldn
did didn do does doesn doing don down during
each
few for from further
had hadn has hasn have haven having he her here hers herself him himself his how
i if in into is isn it its itself
just
ll
m ma me mightn more most mustn my myself
needn no nor not now
o of off on once only or other our ours ourselves out over own
re
s same shan she should shouldn so some such
t than that the their theirs them themselves then there these they this those through
to too
under until up
ve very
was wasn we were weren what when where which while who whom why will with won
would wouldn
y you your yours yourself yourselves
say says said new old says told tells back like get got make made take taken
many much one two three first second us mr mrs ms also
""".split())

perprocess

In [ ]:
_TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z']+")


def normalize_unicode(text: str) -> str: #strip accents
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))


def tokenize(text: str) -> List[str]:# lowercased word tokens, apostrophes stripped from contractions
    text = normalize_unicode(text).lower()
    return [t.replace("'", "") for t in _TOKEN_RE.findall(text)]


def tokenize_for_topics(text: str, min_len: int = 3) -> List[str]:
    return [
        t for t in tokenize(text)
        if t not in STOPWORDS and len(t) >= min_len
    ]


def tokenize_for_emotion(text: str) -> List[str]:
    return tokenize(text)


def read_lines(path: Path) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [ln.strip() for ln in f if ln.strip()]


def read_label_matrix(path: Path) -> np.ndarray:
    """Load the label file as an (N, 6) integer matrix of raw vote counts."""
    rows: List[List[int]] = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            parts = ln.split()
            if not parts:
                continue
            rows.append([int(x) for x in parts])
    arr = np.asarray(rows, dtype=np.float32)
    if arr.shape[1] != 6:
        raise ValueError(
            f"Expected 6 emotion columns, got {arr.shape[1]} in {path}"
        )
    return arr


def normalize_labels(
    raw: np.ndarray, eps: float = 1e-4
) -> np.ndarray:
    raw = raw.astype(np.float64)
    row_sums = raw.sum(axis=1, keepdims=True)
    zero_mask = (row_sums == 0).flatten()
    raw[zero_mask] = 1.0 / raw.shape[1]
    row_sums = raw.sum(axis=1, keepdims=True)
    probs = raw / row_sums

    if eps > 0:
        uniform = np.full_like(probs, 1.0 / probs.shape[1])
        probs = (1 - eps) * probs + eps * uniform
    return probs.astype(np.float32)

class Vocabulary:
    def __init__(self, cfg: EmotionConfig) -> None:
        self.cfg = cfg
        self.itos: List[str] = [cfg.pad_token, cfg.unk_token]
        self.stoi: Dict[str, int] = {tok: i for i, tok in enumerate(self.itos)}
    @property
    def pad_idx(self) -> int:
        return self.stoi[self.cfg.pad_token]

    @property
    def unk_idx(self) -> int:
        return self.stoi[self.cfg.unk_token]

    def __len__(self) -> int:
        return len(self.itos)

    def build(self, token_lists: Iterable[Sequence[str]]) -> "Vocabulary":
        counter: Counter = Counter()
        for toks in token_lists:
            counter.update(toks)

        ordered = sorted(
            counter.items(), key=lambda kv: (-kv[1], kv[0])
        )
        for tok, freq in ordered:
            if freq < self.cfg.min_word_freq:
                break
            if len(self.itos) >= self.cfg.max_vocab_size:
                break
            if tok in self.stoi:
                continue
            self.stoi[tok] = len(self.itos)
            self.itos.append(tok)
        return self

    def encode(self, tokens: Sequence[str]) -> List[int]:
        unk = self.unk_idx
        return [self.stoi.get(t, unk) for t in tokens]

    def encode_padded(
        self, tokens: Sequence[str], max_len: int
    ) -> Tuple[List[int], int]:
        """Truncate or pad to `max_len`. Returns (ids, true_length)."""
        ids = self.encode(tokens)[:max_len]
        true_len = max(1, len(ids))
        if len(ids) < max_len:
            ids = ids + [self.pad_idx] * (max_len - len(ids))
        return ids, true_len



topic mode

In [ ]:
log = logging.getLogger(__name__)

class TopicModel: # thin wrapper to bundle dictionaryt, coherence and model
    def __init__(self, cfg: TopicConfig) -> None:
        self.cfg = cfg
        self.dictionary: Dictionary | None = None
        self.lda: LdaModel | None = None
        self.tokenized_docs: List[List[str]] | None = None
        self.coherence: float | None = None

    def fit(self, raw_docs: Sequence[str]) -> "TopicModel":
        log.info("Tokenizing %d documents for topic modeling", len(raw_docs))
        self.tokenized_docs = [
            tokenize_for_topics(d, min_len=self.cfg.min_token_len)
            for d in raw_docs
        ]

        self.dictionary = Dictionary(self.tokenized_docs)
        n_pre = len(self.dictionary)
        self.dictionary.filter_extremes(
            no_below=self.cfg.no_below,
            no_above=self.cfg.no_above,
        )
        log.info(
            "Filtered dictionary from %d to %d unique tokens (no_below=%d, no_above=%.2f)",
            n_pre, len(self.dictionary), self.cfg.no_below, self.cfg.no_above,
        )

        corpus = [self.dictionary.doc2bow(d) for d in self.tokenized_docs]

        log.info(
            "LDA model: num_topics=%d, passes=%d, iterations=%d",
            self.cfg.num_topics, self.cfg.passes, self.cfg.iterations,
        )
        self.lda = LdaModel(
            corpus=corpus,
            id2word=self.dictionary,
            num_topics=self.cfg.num_topics,
            passes=self.cfg.passes,
            iterations=self.cfg.iterations,
            chunksize=self.cfg.chunksize,
            alpha=self.cfg.alpha,
            eta=self.cfg.eta,
            eval_every=self.cfg.eval_every,
            random_state=self.cfg.random_state,
        )

        self.coherence = self._compute_coherence()
        log.info("Coherence (%s) = %.4f", self.cfg.coherence_metric, self.coherence)
        return self

    def _compute_coherence(self) -> float:
        cm = CoherenceModel(
            model=self.lda,
            texts=self.tokenized_docs,
            dictionary=self.dictionary,
            coherence=self.cfg.coherence_metric,
        )
        return float(cm.get_coherence())


    def top_words_per_topic(self) -> List[List[str]]:
        if self.lda is None:
            raise RuntimeError("TopicModel.fit() must be called first")
        out: List[List[str]] = []
        for topic_id in range(self.cfg.num_topics):
            terms = self.lda.show_topic(topic_id, topn=self.cfg.top_n_words)
            out.append([w for w, _ in terms])
        return out

    def write(self, path: Path) -> None:
        if self.coherence is None:
            raise RuntimeError("TopicModel.fit() must be called first")
        topics = self.top_words_per_topic()
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(f"{self.coherence:.3f}\n")
            for words in topics:
                f.write(" ".join(words) + "\n")
        log.info("Wrote topic results → %s", path)

emotion model

inputs id -> embedding -> -> BiLSTM -> additive attention -> context vector -> (dense -> relu -> dropouyt -> dense -> softmax) -> emotion distro

the above is th rough representation of the neural architecture for the emotion distro prediction.

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim: int, attention_dim: int) -> None:
        super().__init__()
        self.W = nn.Linear(hidden_dim, attention_dim, bias=True)
        self.v = nn.Linear(attention_dim, 1, bias=False)

    def forward(
        self, hidden: torch.Tensor, mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # hidden: (B, T, D),  mask: (B, T)
        scores = self.v(torch.tanh(self.W(hidden))).squeeze(-1)  # (B, T)
        scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)                      # (B, T)
        context = torch.bmm(weights.unsqueeze(1), hidden).squeeze(1)  # (B, D)
        return context, weights

class EmotionClassifier(nn.Module):

    def __init__(self, vocab_size: int, pad_idx: int, cfg: EmotionConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.pad_idx = pad_idx

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=cfg.embedding_dim,
            padding_idx=pad_idx,
        )
        self.emb_dropout = nn.Dropout(cfg.dropout)

        self.lstm = nn.LSTM(
            input_size=cfg.embedding_dim,
            hidden_size=cfg.lstm_hidden,
            num_layers=cfg.lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.dropout if cfg.lstm_layers > 1 else 0.0,
        )

        self.attention = AdditiveAttention(
            hidden_dim=cfg.lstm_hidden * 2,
            attention_dim=cfg.attention_dim,
        )

        self.classifier = nn.Sequential(
            nn.Linear(cfg.lstm_hidden * 2, cfg.lstm_hidden),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.lstm_hidden, cfg.num_classes),
        )

        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.xavier_uniform_(self.embedding.weight)
        with torch.no_grad():
            self.embedding.weight[self.pad_idx].zero_()
        for m in self.classifier:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(
        self, ids: torch.Tensor, lengths: torch.Tensor
    ) -> torch.Tensor:
        mask = (ids != self.pad_idx).float()
        x = self.emb_dropout(self.embedding(ids))

        lengths_cpu = lengths.detach().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths_cpu, batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        hidden, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=ids.size(1)
        )

        context, _ = self.attention(hidden, mask)
        logits = self.classifier(context)
        return logits

def kl_divergence_loss(
    logits: torch.Tensor, targets: torch.Tensor
) -> torch.Tensor:

    log_probs = F.log_softmax(logits, dim=-1)
    return F.kl_div(log_probs, targets, reduction="batchmean")


def soft_cross_entropy_loss(
    logits: torch.Tensor, targets: torch.Tensor
) -> torch.Tensor:
    log_probs = F.log_softmax(logits, dim=-1)
    return -(targets * log_probs).sum(dim=-1).mean()

def mse_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(F.softmax(logits, dim=-1), targets)


def get_loss_fn(name: str):
    return {
        "kl_div": kl_divergence_loss,
        "soft_ce": soft_cross_entropy_loss,
        "mse": mse_loss,
    }[name]



training and inference for the emotion model

In [ ]:
import random
import math
class HeadlineDataset(Dataset):
  def __init__(
        self,
        token_lists: Sequence[Sequence[str]],
        vocab: Vocabulary,
        max_len: int,
        targets: np.ndarray | None = N  one,
    ) -> None:
        self.token_lists = token_lists
        self.vocab = vocab
        self.max_len = max_len
        self.targets = targets
        if targets is not None and len(targets) != len(token_lists):
            raise ValueError("token_lists and targets must align")
  def __len__(self) -> int:
    return len(self.token_lists)
  def __getitem__(self, idx:int):
    ids, length = self.vocab.encode_padded(self.token_lists[idx], self.max_len)
    ids_t = torch.tensor(ids, dtype=torch.long)
    len_t = torch.tensor(length, dtype=torch.long)
    if self.targets is not None:
      tgt = torch.tensor(self.targets[idx], dtype=torch.float32)
      return ids_t, len_t, tgt
    return ids_t, len_t
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@dataclass
class TrainResult:
  model: EmotionClassifier
  best_val_loss: float
  epoch_of_best: int

def _make_loaders(
    train_tokens: List[List[str]],
    train_targets: np.ndarray,
    vocab: Vocabulary,
    cfg: EmotionConfig,
    seed: int,
) -> Tuple[DataLoader, DataLoader]:
    n = len(train_tokens)
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    val_n = max(1, int(round(cfg.val_split * n)))
    val_idx, train_idx = idx[:val_n], idx[val_n:]

    def slice_pair(indices):
        return (
            [train_tokens[i] for i in indices],
            train_targets[indices],
        )

    tr_toks, tr_tgts = slice_pair(train_idx)
    va_toks, va_tgts = slice_pair(val_idx)

    tr_ds = HeadlineDataset(tr_toks, vocab, cfg.max_seq_len, tr_tgts)
    va_ds = HeadlineDataset(va_toks, vocab, cfg.max_seq_len, va_tgts)

    g = torch.Generator()
    g.manual_seed(seed)
    tr_loader = DataLoader(
        tr_ds, batch_size=cfg.batch_size, shuffle=True, generator=g
    )
    va_loader = DataLoader(
        va_ds, batch_size=cfg.batch_size, shuffle=False
    )
    return tr_loader, va_loader


def _run_epoch(
    model: EmotionClassifier,
    loader: DataLoader,
    loss_fn,
    optimizer: torch.optim.Optimizer | None,
    device: torch.device,
    grad_clip: float,
) -> float:
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, total_n = 0.0, 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for ids, lengths, targets in loader:
            ids = ids.to(device)
            lengths = lengths.to(device)
            targets = targets.to(device)

            logits = model(ids, lengths)
            loss = loss_fn(logits, targets)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            bsz = ids.size(0)
            total_loss += loss.item() * bsz
            total_n += bsz
    return total_loss / max(total_n, 1)

def train_single(
    train_tokens: List[List[str]],
    train_targets: np.ndarray,
    vocab: Vocabulary,
    cfg: EmotionConfig,
    device: torch.device,
    seed: int,
) -> TrainResult:
    set_seed(seed)

    model = EmotionClassifier(
        vocab_size=len(vocab),
        pad_idx=vocab.pad_idx,
        cfg=cfg,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )
    loss_fn = get_loss_fn(cfg.loss_fn)

    tr_loader, va_loader = _make_loaders(
        train_tokens, train_targets, vocab, cfg, seed
    )

    best_val = math.inf
    best_state: dict | None = None
    best_epoch = -1
    bad_epochs = 0

    for epoch in range(1, cfg.epochs + 1):
        tr_loss = _run_epoch(
            model, tr_loader, loss_fn, optimizer, device, cfg.grad_clip
        )
        va_loss = _run_epoch(
            model, va_loader, loss_fn, None, device, cfg.grad_clip
        )
        scheduler.step(va_loss)

        if va_loss < best_val - 1e-5:
            best_val = va_loss
            best_state = {
                k: v.detach().cpu().clone() for k, v in model.state_dict().items()
            }
            best_epoch = epoch
            bad_epochs = 0
        else:
            bad_epochs += 1

        if epoch == 1 or epoch % 5 == 0 or bad_epochs == cfg.patience:
            log.info(
                "[seed=%d] ep %02d  train=%.4f  val=%.4f  best=%.4f@%d",
                seed, epoch, tr_loss, va_loss, best_val, best_epoch,
            )

        if bad_epochs >= cfg.patience:
            log.info("[seed=%d] early stop at epoch %d", seed, epoch)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return TrainResult(model=model, best_val_loss=best_val, epoch_of_best=best_epoch)

#begin inference

@torch.no_grad()
def predict_distribution(
    model: EmotionClassifier,
    test_tokens: List[List[str]],
    vocab: Vocabulary,
    cfg: EmotionConfig,
    device: torch.device,
) -> np.ndarray:
    model.eval()
    ds = HeadlineDataset(test_tokens, vocab, cfg.max_seq_len, targets=None)
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False)
    out: List[np.ndarray] = []
    for ids, lengths in loader:
        ids = ids.to(device)
        lengths = lengths.to(device)
        logits = model(ids, lengths)
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        out.append(probs)
    return np.concatenate(out, axis=0)

def train_and_predict(
    raw_train_texts: Sequence[str],
    raw_train_labels: np.ndarray,
    raw_test_texts: Sequence[str],
    cfg: EmotionConfig,
) -> np.ndarray:
    """Full pipeline: build vocab → train ensemble → average predictions."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log.info("Device: %s", device)

    train_tokens = [tokenize_for_emotion(t) for t in raw_train_texts]
    test_tokens = [tokenize_for_emotion(t) for t in raw_test_texts]

    vocab = Vocabulary(cfg).build(train_tokens)
    log.info("Vocabulary size: %d", len(vocab))

    train_targets = normalize_labels(
        raw_train_labels, eps=cfg.label_smoothing_eps
    )

    ensemble_preds: List[np.ndarray] = []
    val_losses: List[float] = []
    for seed in cfg.ensemble_seeds:
        log.info("===== Training seed %d ====", seed)
        result = train_single(
            train_tokens, train_targets, vocab, cfg, device, seed
        )
        val_losses.append(result.best_val_loss)
        preds = predict_distribution(
            result.model, test_tokens, vocab, cfg, device
        )
        ensemble_preds.append(preds)

    avg = np.mean(np.stack(ensemble_preds, axis=0), axis=0)
    avg = avg / avg.sum(axis=1, keepdims=True)

    log.info(
        "Ensemble val losses: %s  (mean=%.4f)",
        ["%.4f" % v for v in val_losses], float(np.mean(val_losses)),
    )
    return avg


def write_emotion_output(predictions: np.ndarray, path: Path) -> None:
    """Write per-row 6-float emotion distributions in the grader format."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in predictions:
            f.write(" ".join(f"{p:.4f}" for p in row) + "\n")
    log.info("Wrote emotion predictions → %s", path)

main block


In [ ]:
import time
import sys
import argparse

def _setup_logging() -> None:
  logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
        stream=sys.stdout,
        force=True,
    )


def run_topic_pipeline() -> Path:
    log = logging.getLogger("topic")
    log.info("Loading training corpus from %s", CFG.paths.train_file)
    train_texts = read_lines(CFG.paths.train_file)
    log.info("Read %d training documents", len(train_texts))

    tm = TopicModel(CFG.topic).fit(train_texts)
    tm.write(CFG.paths.topic_output)

    print("\n" + "=" * 60)
    print(f"TOPIC MODEL  —  C_v coherence = {tm.coherence:.3f}")
    print("=" * 60)
    for i, words in enumerate(tm.top_words_per_topic(), start=1):
        print(f"  Topic {i:>2}: {' · '.join(words)}")
    print("=" * 60 + "\n")
    return CFG.paths.topic_output

def run_emotion_pipeline() -> Path:
    log = logging.getLogger("emotion")
    log.info("Loading data files")
    train_texts = read_lines(CFG.paths.train_file)
    test_texts = read_lines(CFG.paths.test_file)
    train_labels = read_label_matrix(CFG.paths.label_file)

    if len(train_texts) != len(train_labels):
        raise ValueError(
            f"#train texts ({len(train_texts)}) != "
            f"#label rows ({len(train_labels)})"
        )

    log.info(
        "Train=%d  Test=%d  Labels shape=%s",
        len(train_texts), len(test_texts), train_labels.shape,
    )

    preds = train_and_predict(
        raw_train_texts=train_texts,
        raw_train_labels=train_labels,
        raw_test_texts=test_texts,
        cfg=CFG.emotion,
    )
    write_emotion_output(preds, CFG.paths.emotion_output)
    sums = preds.sum(axis=1)
    log.info(
        "Prediction sanity: min_sum=%.4f  max_sum=%.4f  mean_sum=%.4f",
        sums.min(), sums.max(), sums.mean(),
    )
    return CFG.paths.emotion_output


def main() -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--topic-only", action="store_true")
    parser.add_argument("--emotion-only", action="store_true")
    args, unknown = parser.parse_known_args()

    _setup_logging()
    start = time.perf_counter()

    if not args.emotion_only:
        run_topic_pipeline()
    if not args.topic_only:
        run_emotion_pipeline()

    elapsed = time.perf_counter() - start
    print(f"\nPipeline complete in {elapsed:.1f}s")
    print(f"  • {CFG.paths.topic_output}")
    print(f"  • {CFG.paths.emotion_output}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

17:46:00 | INFO    | topic | Loading training corpus from /content/drive/MyDrive/Major Project/train.txt
17:46:01 | INFO    | topic | Read 1000 training documents
17:46:01 | INFO    | __main__ | Tokenizing 1000 documents for topic modeling
17:46:01 | INFO    | gensim.corpora.dictionary | adding document #0 to Dictionary<0 unique tokens: []>
17:46:01 | INFO    | gensim.corpora.dictionary | built Dictionary<2707 unique tokens: ['approved', 'breast', 'cancer', 'predict', 'relapse']...> from 1000 documents (total 4862 corpus positions)
17:46:01 | INFO    | gensim.utils | Dictionary lifecycle event {'msg': "built Dictionary<2707 unique tokens: ['approved', 'breast', 'cancer', 'predict', 'relapse']...> from 1000 documents (total 4862 corpus positions)", 'datetime': '2026-05-09T17:46:01.973816', 'gensim': '4.4.0', 'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'event': 'created'}
17:46:01 | INFO    | gensim.corpora.dictionar

SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
